# SAE latent MUC (Colab)

Отдельный пайплайн из папки `sae_muc/`: интервенция через **encode → сдвиг латентов → decode + error term**.

Репозиторий **sae-muc** содержит только `sae_muc/` и Colab-ноутбуки. Данные VUF (`datasets/`, `detection/`, `calibration/` с `Hs_hedge_universal.pt`) скопируйте **в корень клона** (`REPO_DIR`, по умолчанию `/content/sae-muc`) или положите рядом и задайте `--repo_root`.

**Нужны:** GPU, данные (`datasets/`, `detection/`, `calibration/outputs/.../Hs_hedge_universal.pt`), Hugging Face токен для Mistral.

Ниже — периодическое копирование выбранных путей из `/content` на Google Drive (фоновый поток).

## 0. GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Runtime → GPU"

## 1. Клон репозитория и зависимости

In [ ]:
import os, sys, subprocess

GIT_URL = os.environ.get("SAE_MUC_GIT_URL", "https://github.com/SadreevAmir/sae-muc.git")
GIT_BRANCH = os.environ.get("SAE_MUC_BRANCH", "main")
REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")

sae_pkg = os.path.join(REPO_DIR, "sae_muc")
if not os.path.isdir(sae_pkg):
    parent = os.path.dirname(REPO_DIR.rstrip("/")) or "/content"
    os.makedirs(parent, exist_ok=True)
    if os.path.isdir(REPO_DIR):
        subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

assert os.path.isdir(sae_pkg), f"После клона ожидается {sae_pkg}"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!pip install -q -r sae_muc/requirements.txt


## 2. Google Drive: монтирование и периодический бэкап

Задайте `DRIVE_BACKUP_ROOT` и список пар **(источник в /content → папка на Drive)**. Каждые `BACKUP_INTERVAL_SEC` секунд файлы копируются (сразу после старта — первая копия, затем по таймеру).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_BACKUP_ROOT = "/content/drive/MyDrive/vuf_sae_muc_backup"  # <-- свой путь
BACKUP_INTERVAL_SEC = 600  # 10 минут

import os
from pathlib import Path

REPO_DIR = "/content/sae-muc"
pairs = [
    (f"{REPO_DIR}/sae_muc/outputs", f"{DRIVE_BACKUP_ROOT}/sae_muc/outputs"),
    (f"{REPO_DIR}/sae_muc/artifacts", f"{DRIVE_BACKUP_ROOT}/sae_muc/artifacts"),
]

Path(DRIVE_BACKUP_ROOT).mkdir(parents=True, exist_ok=True)

import sys
sys.path.insert(0, REPO_DIR)
from sae_muc.drive_sync import PeriodicDriveBackup

_drive_backup = PeriodicDriveBackup(pairs, interval_sec=BACKUP_INTERVAL_SEC)
_drive_backup.start()
print("Периодический бэкап на Drive запущен.")
print("Остановка: _drive_backup.stop()")

## 3. (Опционально) Чекпоинт фаз 1–6 с Drive → репо

Если у вас есть папка `vuf_checkpoint_phase6` на Drive (как в `vuf_checkpoint/README.md`), скопируйте `datasets`, `detection`, `calibration` в `REPO_DIR`.

In [ ]:
import shutil
from pathlib import Path

REPO_DIR = Path("/content/sae-muc")
CHECKPOINT_SRC = Path("/content/drive/MyDrive/vuf_checkpoint_phase6")  # или None

if CHECKPOINT_SRC and CHECKPOINT_SRC.is_dir():
    for name in ("datasets", "detection", "calibration"):
        src = CHECKPOINT_SRC / name
        dst = REPO_DIR / name
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print("Copied", name)
else:
    print("Пропуск копирования чекпоинта (нет CHECKPOINT_SRC).")

## 4. Hugging Face login

In [ ]:
from huggingface_hub import login
login()  # токен с доступом к Mistral

## 5. Сборка `intervention.pt` из Hs_hedge

In [ ]:
import subprocess, sys

REPO_DIR = "/content/sae-muc"
HEDGE = f"{REPO_DIR}/calibration/outputs/merged/Mistral-7B-Instruct-v0.3/uncertainty/Hs_hedge_universal.pt"
OUT = f"{REPO_DIR}/sae_muc/artifacts/mistral_intervention.pt"

cmd = [
    sys.executable, "-m", "sae_muc.build_intervention_config",
    "--hedge_path", HEDGE,
    "--out_path", OUT,
    "--release", "mistral-7b-res-wg",
    "--top_k", "64",
    "--sae_device", "cpu",
]
subprocess.check_call(cmd, cwd=REPO_DIR)

## 6. Запуск SAE MUC

In [ ]:
import subprocess, sys

REPO_DIR = "/content/sae-muc"
# Имя jsonl как у semantic_control: with_vufi_2_range(15,32)_1.0.jsonl
# Чтобы сразу гонять eval из фазы 10.3, задайте OUTPUT_DIR под calibration/outputs/.../test
OUTPUT_DIR = None  # например: f"{REPO_DIR}/calibration/outputs/nq_open/Mistral-7B-Instruct-v0.3/uncertainty/test"

cmd = [
    sys.executable, "-m", "sae_muc.run_muc",
    "--repo_root", REPO_DIR,
    "--dataset", "nq_open",
    "--split", "test",
    "--model_name", "Mistral-7B-Instruct-v0.3",
    "--prompt_type", "uncertainty",
    "--str_process_layers", "range(15,32)",
    "--intervention_path", f"{REPO_DIR}/sae_muc/artifacts/mistral_intervention.pt",
    "--max_alpha", "1.0",
]
if OUTPUT_DIR:
    cmd += ["--output_dir", OUTPUT_DIR]
subprocess.check_call(cmd, cwd=REPO_DIR)

## 7. Остановить бэкап (опционально)

In [ ]:
# _drive_backup.stop()